# Time diagnostics
Create one PDF for energy/wave action and one for dissipation and injection rates. Select a time or frame interval and optionally apply a centered rolling average.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

SCRIPT_DIRECTORY = Path.cwd() if (Path.cwd() / 'gp2d_plotting.py').exists() else Path.cwd() / 'scripts'
sys.path.insert(0, str(SCRIPT_DIRECTORY.resolve()))
from gp2d_plotting import (filter_time_series, read_csv, repository_root,
    rolling_mean, save_figure, use_plot_style)

## Configuration

In [ ]:
ROOT = repository_root()  # Repository root; normally no change is needed.
DIAGNOSTICS_FILE = ROOT / 'output/diagnostics.csv'  # CSV produced by the solver.
QUANTITIES_FIGURE = ROOT / 'figures/diagnostics_quantities.pdf'  # Energy/action PDF.
DISSIPATION_FIGURE = ROOT / 'figures/diagnostics_dissipation.pdf'  # Rate PDF.

# Inclusive time interval, e.g. (0.1, 1.0). None keeps every saved time.
# Use None for either bound, e.g. (0.1, None), to leave that side open.
TIME_RANGE = None
# Inclusive frame interval, e.g. (100, 1000). None keeps every frame.
# When both ranges are set, samples must satisfy both selections.
FRAME_RANGE = None
# Number of saved samples in the centered moving average. 1 plots raw data.
# Values >1 smooth every curve and leave NaNs at the incomplete edge windows.
ROLLING_WINDOW = 1
# True divides energies, wave action, dissipation, and injection by DOMAIN_AREA.
NORMALIZE_BY_AREA = False
# Physical area used for normalization. Change this for a non-2pi by 2pi domain.
DOMAIN_AREA = (2.0 * np.pi) ** 2
USE_TEX = True      # True uses an external LaTeX installation for all figure text.
FONT_SIZE = 15      # Base font size in points.

In [ ]:
use_plot_style(USE_TEX, FONT_SIZE)
table = filter_time_series(read_csv(DIAGNOSTICS_FILE), TIME_RANGE, FRAME_RANGE)
time = table['time']
normalization = DOMAIN_AREA if NORMALIZE_BY_AREA else 1.0

def series(column):
    return rolling_mean(table[column] / normalization, ROLLING_WINDOW)

print(f'Using {table.size} samples, frames {int(table["frame"][0])}--{int(table["frame"][-1])}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].plot(time, series('total_energy'), label=r'$H$ (total)', color='black')
axes[0].plot(time, series('kinetic_energy'), label=r'$H_{\rm kin}$')
axes[0].plot(time, series('potential_energy'), label=r'$H_{\mu}$')
axes[0].plot(time, series('nonlinear_energy'), label=r'$H_{\rm nl}$')
axes[1].plot(time, series('wave_action'), label=r'$N$', color='C0')
axes[0].set_ylabel('energy density' if NORMALIZE_BY_AREA else 'energy')
axes[1].set_ylabel('wave-action density' if NORMALIZE_BY_AREA else 'wave action')
for axis in axes:
    axis.set_xlabel(r'$t$')
    axis.grid(True, alpha=0.2)
    axis.legend()
saved = save_figure(fig, QUANTITIES_FIGURE)
print(f'Wrote {saved}')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
wave_hypo = series('wave_action_dissipation_hypo')
wave_hyper = series('wave_action_dissipation_hyper')
axes[0].plot(time, wave_hypo, label='hypoviscous')
axes[0].plot(time, wave_hyper, label='hyperviscous')
axes[0].plot(time, wave_hypo + wave_hyper, label='total', color='black', linestyle='--')
axes[0].set_ylabel(r'dissipation rate of $N$')

energy_hypo = series('total_energy_dissipation_hypo')
energy_hyper = series('total_energy_dissipation_hyper')
axes[1].plot(time, energy_hypo, label='hypoviscous')
axes[1].plot(time, energy_hyper, label='hyperviscous')
axes[1].plot(time, energy_hypo + energy_hyper, label='total', color='black', linestyle='--')
axes[1].plot(time, series('quadratic_energy_dissipation_hypo') +
                  series('quadratic_energy_dissipation_hyper'),
             label='quadratic-only dissipation', linestyle='-.')
axes[1].set_ylabel(r'dissipation rate of $H$')
axes[2].plot(time, series('expected_full_energy_injection'), color='C2', label='expected injection')
axes[2].set_ylabel(r'expected injection rate of $H$')
for axis in axes:
    axis.set_xlabel(r'$t$')
    axis.grid(True, alpha=0.2)
    axis.legend()
saved = save_figure(fig, DISSIPATION_FIGURE)
print(f'Wrote {saved}')
plt.show()